In [0]:

dbutils.widgets.text("path","")

In [0]:
dbutils.widgets.text("path","")

from datetime import date
from common_utils.logging import get_logger
from common_utils.ingestor import read_jdbc, write_raw
import json


# ============================================================
# 0. LOGGER
# ============================================================
logger = get_logger("sql-server-ingestion")

# ============================================================
# 1. CONFIG PATH
# ============================================================

dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")


# ============================================================
# 2. READ CONFIG
# ============================================================
with open(config_path, "r") as f:
    config = json.load(f)

# ============================================================
# 3. CONFIG SECTIONS
# ============================================================

source_config = config["source"]
target_config = config["target"]
write_options = config["write_options"]

# ============================================================
# 4. RUN DATE
# ============================================================
run_date = date.today().isoformat()
logger.info("Load date is %s",run_date )
logger.info("Reading data from  %s",source_config["dbtable"] )




user = dbutils.secrets.get(scope="retail-platform-dev", key = source_config["user"])
password = dbutils.secrets.get(scope="retail-platform-dev", key = source_config["password"])

# ============================================================
# 5. READ FROM SQL SERVER
# ============================================================
df = read_jdbc(spark, source_config["url"],source_config["dbtable"], user, password  )
logger.info("read %s rows", df.count())
logger.info("sample data.......")
df.show()


# ============================================================
# 6. TARGET PATH
# ============================================================

target_path = f"{target_config['base_path']}/{target_config['folder']}/load_date={run_date}"
logger.info("writiing data to %s", target_path)

# ============================================================
# 7. WRITE RAW
# ============================================================
write_raw(df, target_path,target_config["file_format"], target_config["mode"], write_options)
logger.info("data landed at %s", target_path)

2026-10-28 12:01:58,505 INFO Load date is 2026-10-28
2026-10-28 12:01:58,506 INFO Reading data from  retail.customers
2026-10-28 12:02:03,244 INFO read 28813 rows
2026-10-28 12:02:03,245 INFO sample data.......
2026-10-28 12:02:05,973 INFO writiing data to /Volumes/retaildata_platform/bronze/raw_data//sqlserver_customers/load_date=2026-10-28


+-----------+---------+--------+--------------------+-----+-------------------+----------+--------------+----------+----+-------+-----------+------------------+-----------------+--------------------+----------+----------+---------------+---------------+
|customer_id|   tax_id|tax_code|       customer_name|state|               city|  postcode|        street|    number|unit| region|   district|               lon|              lat|     ship_to_address|valid_from|  valid_to|units_purchased|loyalty_segment|
+-----------+---------+--------+--------------------+-----+-------------------+----------+--------------+----------+----+-------+-----------+------------------+-----------------+--------------------+----------+----------+---------------+---------------+
|   11123757|     NULL|    NULL|     SMITH,  SHIRLEY|   IN|             BREMEN|   46506.0|   N CENTER ST|     521.0|NULL|Indiana|       50.0|       -86.1465825|       41.4507625|IN, 46506.0, N CE...|1532824233|1548137353|             34| 

2026-10-28 12:02:13,414 INFO data landed at /Volumes/retaildata_platform/bronze/raw_data//sqlserver_customers/load_date=2026-10-28
